
# Module 24: Principal Component Analysis (PCA)

This notebook shows PCA **using a library** (scikit-learn) on a **small dataset**, with detailed explanations for each step.

### What you will learn here
You will learn how to:
1. Understand *why* PCA is useful in practice (beyond theory)
2. Prepare data properly (especially **scaling**, which is not optional most of the time)
3. Fit PCA and interpret **explained variance ratio**
4. Choose the number of components using a clear rule (for example, 90% or 95% variance)
5. Transform the dataset into PCA space
6. Visualize the data in 2D using PC1 and PC2
7. Compare model performance **with and without** PCA (small demo)

### Key idea to keep in mind
PCA does **not** select original features. It creates **new features** (principal components) that are combinations of the original ones.



## 1) Imports and Setup

We will use:
- `sklearn.datasets` for a small dataset
- `StandardScaler` for scaling
- `PCA` for dimensionality reduction
- `matplotlib` for plotting

Why scaling?  
PCA is variance-based. If one feature has a larger numeric scale, it can dominate the variance and hijack PCA. Scaling makes features comparable so PCA captures structure, not unit differences.


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_wine
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report


## 2) Load a Small Dataset

We will use the **Wine dataset** (classic, small, and perfect for PCA demos):
- 178 samples
- 13 numeric features
- 3 classes

Why this dataset is good for PCA:
- Features are on different scales (some are small decimals, some are larger)
- Many features are correlated
- PCA often produces a nice 2D visualization


In [2]:
data = load_wine()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name="class")

X.shape, y.shape

((178, 13), (178,))

In [3]:
X.head()

,alcohol,malic_acid,ash,alcalinity_of_ash,magnesium,total_phenols,flavanoids,nonflavanoid_phenols,proanthocyanins,color_intensity,hue,od280/od315_of_diluted_wines,proline
0,14.23,1.71,2.43,15.6,127.0,2.80,3.06,0.28,2.29,5.64,1.04,3.92,1065.0
1,13.20,1.78,2.14,11.2,100.0,2.65,2.76,0.26,1.28,4.38,1.05,3.40,1050.0
2,13.16,2.36,2.67,18.6,101.0,2.80,3.24,0.30,2.81,5.68,1.03,3.17,1185.0
3,14.37,1.95,2.50,16.8,113.0,3.85,3.49,0.24,2.18,7.80,0.86,3.45,1480.0
4,13.24,2.59,2.87,21.0,118.0,2.80,2.69,0.39,1.82,4.32,1.04,2.93,735.0



## 3) Quick Data Understanding: Feature Scales

Before PCA, always check scales.
If features are on very different ranges, PCA will mainly focus on the largest-scale feature.

Let's look at descriptive stats quickly.


In [4]:
X.describe().T[["mean", "std", "min", "max"]].head(10)

,mean,std,min,max
alcohol,13.000618,0.811827,11.03,14.83
malic_acid,2.336348,1.117146,0.74,5.80
ash,2.366517,0.274344,1.36,3.23
alcalinity_of_ash,19.494944,3.339564,10.60,30.00
magnesium,99.741573,14.282484,70.00,162.00
total_phenols,2.295112,0.625851,0.98,3.88
flavanoids,2.029270,0.998859,0.34,5.08
nonflavanoid_phenols,0.361854,0.124453,0.13,0.66
proanthocyanins,1.590899,0.572359,0.41,3.58
color_intensity,5.058090,2.318286,1.28,13.00
